# Une contrainte d'énergie n'est pas une rigidité

*Pourquoi un modèle linéaire en prévision parfaite retrouve le niveau moyen des prix mais pas leur forme, et quels prix de modèle servent à quelle question.*

Ce notebook est tiré du texte [en ligne](https://www.robingirard.eu/assets/parcours-es/latest/textes/forme-des-prix.html) du parcours ES ; exécutez les cellules dans l'ordre (Maj + Entrée). Les commentaires des cellules donnent les valeurs attendues.

**La cellule d'installation.** En local, placez ce notebook dans le dépôt `pommes-case-studies` (à sa racine ou dans un de ses dossiers) et ouvrez-le avec le noyau du venv du dépôt (`.venv`) : la cellule n'installe rien, elle trouve le dépôt et affiche les versions des paquets.

Sur Google Colab (Fichier › Importer un notebook), la cellule clone le dépôt et installe les paquets : 2 à 3 minutes, puis elle s'arrête volontairement sur `RedemarrezLaSession`. Faites Exécution › Redémarrer la session, relancez cette cellule, et seulement elle, puis continuez dans l'ordre.

In [ ]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

VERSIONS = {"pommes": "0.11.0", "pommes-craft": "0.6.0", "linopy": "0.10.1", "highspy": "1.15.1"}


def versions_du_depot() -> bool:
    """Vrai si les paquets installés sont ceux de requirements.txt (les valeurs attendues en dépendent)."""
    try:
        return all(version(nom) == v for nom, v in VERSIONS.items())
    except PackageNotFoundError:
        return False


class RedemarrezLaSession(Exception):
    """Colab : les paquets sont installés, mais la session garde en mémoire les anciens numpy et pandas."""


if "google.colab" in sys.modules:                     # sur Colab : cloner, installer, redémarrer une fois
    CLONE = Path("/content/pommes-case-studies")
    if not CLONE.exists():
        !git clone https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies {CLONE}
    %cd {CLONE}
    if not versions_du_depot():
        !python --version
        !pip install -q -r requirements.txt
        raise RedemarrezLaSession("Installation finie. Exécution › Redémarrer la session, "
                                  "puis relancez cette cellule (et seulement elle).")

RACINE = next((d for d in (Path.cwd(), *Path.cwd().parents, Path.cwd() / "pommes-case-studies")
               if (d / "pommes_cases").is_dir()), Path.cwd())  # la racine du dépôt des cas
if not (RACINE / "pommes_cases").is_dir():
    raise FileNotFoundError(f"pas de pommes_cases/ sous {RACINE} : placez ce notebook dans le dépôt "
                            "pommes-case-studies (à sa racine ou dans un de ses dossiers)")
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))
print("dépôt :", RACINE.name, "· Python", sys.version.split()[0])
print(" · ".join(f"{nom} {version(nom)}" for nom in VERSIONS))
if not versions_du_depot():
    print("Attention : ce ne sont pas les versions du dépôt", VERSIONS, ": les valeurs attendues peuvent bouger.")

## Le niveau est bon, la forme a disparu

Un modèle fondamental rejoue 2024 : parc, météo, demande, gaz à 38 €/MWh PCI et CO2 à 65 €/t observés, 16 zones, 8 760 heures, programmation linéaire en prévision parfaite. En Allemagne, le prix moyen sort à 83 €/MWh contre 79 sur le marché.

Sur le marché, le profil journalier des prix, rapporté à la moyenne annuelle, dessine deux creux (la nuit, et midi avec le solaire) et deux pointes, le matin et le soir, jusqu'à 1,55 fois la moyenne. Le modèle donne une droite entre 0,95 et 1,05. Marché contre modèle : écart-type des prix horaires 53 €/MWh contre 6 ; heures sous 20 €/MWh, 12 % contre 0 % ; France et Allemagne au même prix, 22 % des heures contre 80 %. Chiffres des planches du cours et d'une étude en cours au centre PERSEE (prix day-ahead EPEX via Energy-Charts, productions ENTSO-E).

Évalués sur ces prix, une batterie ou un électrolyseur ne gagnent presque rien à choisir leurs heures : le facteur de profil d'un électrolyseur qui tourne la nuit et à midi vaut 0,83 sur le marché, 0,98 dans le modèle.

## Le prix est le coût de l'unité qui baisse

Prenez les 1 055 heures de 2024 où le prix allemand passe sous 20 €/MWh. Nucléaire, fossile et lacs totalisent presque autant en réalité que dans le modèle : 112 GW contre 110. Mais la réalité garde 38 GW de fossile et 16 GW de lacs en marche et module le nucléaire de 14 GW. Le modèle fait l'inverse : ordre de mérite pur, le nucléaire à 15,6 €/MWh module en dernier.

Or le prix d'une heure est le coût de l'unité qui baisserait si la demande baissait d'un MW. Sur le marché, à ces heures, c'est le nucléaire qui module ou l'éolien qui s'écrête : près de zéro. Dans le modèle, c'est toujours une centrale au charbon ou au gaz, de 78 à 90 €/MWh, ou un lac dont la valeur de l'eau s'est alignée sur elle. Le fossile flexible des 16 zones ne descend jamais sous 0,8 GW.

## Ce que le programme linéaire ne représente pas

| Ce qui manque | Dans le modèle | Effet sur les prix |
|---|---|---|
| Minimum technique, coût de démarrage | Une centrale passe de 100 % à 0 sans coût | Cause principale : pas d'heures bon marché, pas de prime du soir |
| Débit minimal des lacs | L'eau vaut le cycle combiné au gaz, 88 à 90 €/MWh | Les lacs plafonnent la distribution des deux côtés |
| Modulation réelle du nucléaire | Jamais marginal : 0,3 % des heures en France | La CRE lui attribue le prix 31 % des heures en 2025 |
| Échanges | Capacités fixes, sans flow-based | Couplage trop fréquent, forme inchangée |

Ce sont des contraintes en nombres entiers (marche ou arrêt), que les modèles continentaux omettent pour rester résolubles. Une rampe n'y change presque rien : elle limite une vitesse, la rigidité qui manque est un niveau.

## Le budget nucléaire : 15,6 + 74 = 90

Pour le faire moduler comme en réalité, on impose au nucléaire sa production réelle, mois par mois. Il module bien ses 44 TWh, mais le combustible devient un stock. Le dual du budget, valeur d'un MWh gardé pour plus tard, vaut 74 €/MWh ; ajouté au coût variable de 15,6, il redonne 90 €/MWh, le coût du cycle combiné au gaz. Le nucléaire devient un lac de plus ; l'écart-type des prix tombe à 1,4 €/MWh.

C'est la logique de la valeur de l'eau : en prévision parfaite, avec un fossile toujours disponible, chaque stock vaut le fossile qu'il remplace. Une contrainte d'énergie déplace la production ; les heures bon marché viennent d'unités qui doivent produire quel que soit le prix.

## Exercice : lire le dual d'un budget dans pommes_craft

`ConversionTechnology` de pommes_craft 0.6.0 porte un paramètre `max_yearly_production`, plafond d'énergie annuel qui devient la contrainte `operation_conversion_yearly_production_max_constraint` de pommes 0.11.0. Le « lac » : technologie sans coût variable, capacité fixée, budget. Face à lui, un cycle combiné au gaz à 90 €/MWh et une demande sinusoïdale de 26 à 54 GW. Environnement épinglé du dépôt des cas ; environ une seconde en tout.

In [ ]:
import sys, numpy as np, polars as pl
from pathlib import Path
from pommes_craft import Area, ConversionTechnology, Demand, EconomicHypothesis, LoadShedding, Spillage, TimeStepManager
from pommes_craft.core.model import EnergyModel
from pommes_cases.solve import solve_with_extra_constraints
from pommes.utils import dual_scale

H, Y = 8760, 2024
h = np.arange(H)
demande = 40_000 + 8_000 * np.sin(2 * np.pi * (h % 24 - 9) / 24) + 6_000 * np.cos(2 * np.pi * h / H)  # MW
COUT_LAC, MUST_RUN, BUDGET_TWH = 0.0, np.nan, 120.0        # variante b : 15.6, nan, 120 ; variante c : 15.6, 0.9, 250
frame = lambda v, col: pl.DataFrame({"hour": h, "year_op": np.full(H, Y), col: v}).with_columns(pl.col("hour", "year_op").cast(pl.Int64))

model = EnergyModel(name="lac_budget", folder=Path("sortie"), hours=list(h), year_ops=[Y], year_invs=[Y],
                    year_decs=[Y + 1000], modes=[], resources=["elec"])
with model.context():
    EconomicHypothesis("eco", discount_rate=0.0, year_ref=Y, planning_step=1)
    TimeStepManager("pas", time_step_duration=1.0, operation_year_duration=float(H))
    zone = Area("zone")
    zone.add_component(Demand("demande", resource="elec", demand=frame(demande, "demand")))
    zone.add_component(LoadShedding("delestage", resource="elec", cost=3000.0))
    zone.add_component(Spillage("ecretement", resource="elec", cost=0.0, max_capacity=1e6))   # nan serait lu 0 par craft 0.6.0
    zone.add_component(ConversionTechnology("gaz", factor={"elec": 1.0}, variable_cost=90.0, life_span=1000,
                                            power_capacity_investment_min=60_000, power_capacity_investment_max=60_000))
    zone.add_component(ConversionTechnology("lac", factor={"elec": 1.0}, variable_cost=COUT_LAC, life_span=1000,
                                            power_capacity_investment_min=30_000, power_capacity_investment_max=30_000,
                                            max_yearly_production=BUDGET_TWH * 1e6, must_run=MUST_RUN))

params = {}
lp = solve_with_extra_constraints(model, lambda lp, p: params.setdefault("p", p))   # le crochet garde les paramètres
scale = float(dual_scale(params["p"]).sel(year_op=Y))                              # 1.0 ici : taux nul, 8 760 h
prix = np.round(lp.dual["operation_adequacy_constraint"].sel(area="zone", resource="elec").values.ravel() / scale, 6) + 0.0   # + 0.0 efface le zéro négatif
valeur = -lp.dual["operation_conversion_yearly_production_max_constraint"].sel(area="zone", conversion_tech="lac").squeeze().item() / scale
print(f"valeur du budget : {valeur:.1f} €/MWh ; coût variable + valeur = {COUT_LAC + valeur:.1f} (gaz : 90)")
print(f"prix : moyen {prix.mean():.1f}, min {prix.min():.1f}, max {prix.max():.1f}, écart-type {prix.std():.1f} ; heures sous 20 : {(prix < 20).sum()}")

Résultats avec pommes 0.11.0, pommes_craft 0.6.0, linopy 0.10.1 et HiGHS :

```text
a) lac à 0 €/MWh, budget 120 TWh      valeur du budget : 90.0 ; prix : moyen 90.0, min 90.0, max 90.0, écart-type 0.0 ; heures sous 20 : 0
b) coût 15,6 €/MWh, budget 120 TWh    valeur du budget : 74.4 ; coût variable + valeur = 90.0 ; prix : écart-type 0.0 ; heures sous 20 : 0
c) idem + must_run 0,9, budget 250 TWh valeur du budget : 74.4 ; prix : moyen 88.1, min 0.0, max 90.0, écart-type 12.9 ; heures sous 20 : 184
```

En a, la valeur de l'eau s'aligne sur le gaz : prix plat à 90 €/MWh toutes les heures, alors que la demande varie du simple au double. En b, 15,6 + 74,4 = 90 : le budget nucléaire en petit. En c, le lac doit tourner à 27 GW au moins : les 184 heures où la demande est plus basse passent à zéro. La rigidité crée les heures bon marché ; le budget ne les a jamais créées.

Trois détails. Le dual d'une contrainte « inférieur ou égal » sort négatif : on prend l'opposé. Un prix se lit divisé par `pommes.utils.dual_scale`, qui vaut 1 ici mais pas sur plusieurs années météo. Le crochet de `solve_with_extra_constraints` récupère ici les paramètres ; dans les cas A et D, il ajoute des contraintes avant la résolution.

## 2030 : le creux de midi arrive, la soirée s'efface

Avec le parc National Trends+ du TYNDP 2026, le modèle produit un vrai excédent solaire : 144 à 161 TWh écrêtés sur 16 zones, 14 à 24 % des heures allemandes sous 20 €/MWh. Le creux de midi apparaît, aussi profond qu'aujourd'hui sur le marché. Mais la nuit coûte autant que le soir : rapport nuit/soir de 0,95 dans le modèle, 0,65 sur le marché. Au-dessus du surplus, l'offre n'a qu'une marche : le gaz et les stocks alignés sur lui. Le modèle de l'ENTSO-E a le même profil ; deux modèles d'accord entre eux ne prouvent rien sur la forme.

## Quels prix pour quelle question

| Ce qu'on évalue | Ce qui compte | Prix de modèle 2030 | Ce qu'il faut |
|---|---|---|---|
| Batterie, effacement, stockage journalier | L'écart dans la journée | Sous-estimé d'environ moitié | Des prix à la forme du marché ; prévoir 24 h suffit |
| Électrolyseur qui choisit ses heures | Le prix des heures les moins chères | Du même ordre que le marché en DE et NL | Les prix de modèle conviennent, plutôt prudents |
| Stockage de plusieurs jours, cavité d'hydrogène | Les écarts entre jours et semaines | Du même ordre que le marché | Exploitant myope et valeurs d'usage |
| Réseau, adéquation | Les pointes, les écarts entre zones | Haut de la distribution absent | Beaucoup d'années météo, plusieurs conduites |

Plus la valeur cherchée tient à la journée, plus il faut corriger la forme des prix ; plus elle tient au temps long, plus il faut corriger la prévision. Sur un cas TYNDP 2050, une étude en cours au centre PERSEE mesure qu'un exploitant myope coûte environ 3 % de plus que la prévision parfaite en médiane (chiffre provisoire).

## Vous l'avez compris quand…

- Vous retrouvez 74,4 €/MWh en variante b et dites en une phrase pourquoi 15,6 + 74,4 vaut le coût du gaz.
- Avant de lancer la variante c, vous prédisez le nombre d'heures à zéro : demande sous 0,9 fois 30 GW.
- Pour une batterie et une cavité d'hydrogène, vous dites quoi corriger : la forme des prix ou la prévision.
- Vous expliquez pourquoi des prix « plus variables » ne suffisent pas à mieux évaluer un stockage saisonnier.

## Pour aller plus loin

- Fenêtre d'exploitation et valeurs d'usage, doc POMMES [Fenêtre d'exploitation : capacités fixées, stocks aux bornes, valeurs d'usage](https://pages.persee.minesparis.psl.eu/energy-alternatives/pommes/methodology/operation_window.html) : https://pages.persee.minesparis.psl.eu/energy-alternatives/pommes/methodology/operation_window.html
- Planches remises en séance : KKT et dualité, du multiplicateur de l'équilibre au prix KKT, dualité de Lagrange et prix de marché : exploitation, planification, prix locaux, nodal vs zonal (remis en cours) ; contraintes techniques des centrales, démarrage, minimum technique, modulation du nucléaire Contraintes techniques des centrales : démarrage, durée minimale, minimum technique, rendement à charge partielle, flexibilité du nucléaire (remis en cours).
- Cas A : revenu d'arbitrage d'une batterie sous plusieurs jeux de prix [Cas A — Batteries : que rapporte l'arbitrage au n-ième GW ?](https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies/-/tree/main/A_batteries) : https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies

## Sources

1. Planches du cours « Forme des prix et modélisation » (six planches et notes d'orateur, remises en séance)
2. [CRE, La surveillance et le fonctionnement des marchés de gros de l'électricité et du gaz naturel en 2025](https://www.cre.fr/fileadmin/Documents/Rapports_et_etudes/2026/Rapport_de_surveillance_2025.pdf)
3. [ENTSO-E Transparency Platform (production réalisée par filière, prix day-ahead)](https://transparency.entsoe.eu/)
4. [Energy-Charts (Fraunhofer ISE), prix day-ahead EPEX SPOT par zone](https://www.energy-charts.info/charts/price_spot_market/chart.htm)
5. [ENTSO-E et ENTSOG, scénarios du TYNDP 2026 (National Trends+)](https://2026.entsos-tyndp-scenarios.eu/)
6. [Documentation POMMES, description du modèle (contrainte d'adéquation, plafond annuel de production)](https://pages.persee.minesparis.psl.eu/energy-alternatives/pommes/methodology/model_description.html)
7. [Documentation POMMES, fenêtre d'exploitation et valeurs d'usage](https://pages.persee.minesparis.psl.eu/energy-alternatives/pommes/methodology/operation_window.html)
8. [Dépôt pommes-case-studies (pommes_cases.solve.solve_with_extra_constraints, environnement épinglé)](https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies)